# BSDT H9 — Scaling Test (n=1500, 2000, 2500)

**Key question**: Does WalkSAT effort stay O(1) as n grows?

Known results from H8a:
- n=1000: Stage1=67%, violations≈1.1, WalkSAT 33/33 in 0s → **100%**

We track: violations per failure, WalkSAT flips, |V*|, total time.

Author: Odeyemi Olusegun Israel

In [ ]:
import subprocess, sys
try:
    import numba
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numba', '-q'])

import torch, numpy as np, time, math
import matplotlib.pyplot as plt
from numba import njit
from concurrent.futures import ThreadPoolExecutor, as_completed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
HAS_COMPILE = hasattr(torch, 'compile')
print(f'torch.compile: {"yes" if HAS_COMPILE else "no"}')

In [ ]:
# ══════════════════════════════════════════════
# ENGINE + SCHEDULE + SOLVERS (all-in-one)
# ══════════════════════════════════════════════

def precompute_schedule(fn, steps):
    t = np.linspace(0, 1, steps)
    return torch.from_numpy(np.array([fn(x) for x in t], dtype=np.float32)).to(device)

def sched_delay70(t):
    if t < 0.7: return 0.0
    t2 = (t - 0.7) / 0.3
    return (1.0 - math.cos(math.pi * t2)) / 2.0


class BSDTSonarEngine:
    def __init__(self, n, ni=50, np_=1000, alpha=3.0, mu_scale=0.1):
        self.n, self.ni, self.np_ = n, ni, np_
        self.alpha, self.mu_scale = alpha, mu_scale
        self.m = int(alpha * n)

    def generate_instances(self):
        ni, m, n = self.ni, self.m, self.n
        rand = torch.rand(ni * m, n, device=device)
        cv = rand.argsort(dim=1)[:, :3].reshape(ni, m, 3).long()
        cs = torch.randint(0, 2, (ni, m, 3), device=device).float() * 2 - 1
        return cv, cs

    def compute_mu(self, cv):
        deg = torch.zeros(self.ni, self.n, device=device)
        ones = torch.ones(self.ni, self.m, device=device)
        for p in range(3):
            deg.scatter_add_(1, cv[:, :, p], ones)
        lmax = 0.25 * deg.max(dim=1).values
        return (self.mu_scale * lmax).clamp(min=0.01), lmax

    def energy_and_grad(self, s, cv, cs, mu):
        ni, np_, m, n = self.ni, s.shape[1], self.m, self.n
        cv4 = cv.unsqueeze(1).expand(ni, np_, m, 3)
        s_at = torch.gather(s.unsqueeze(2).expand(ni, np_, m, n), 3, cv4)
        cs4 = cs.unsqueeze(1).expand(ni, np_, m, 3)
        lit = (1.0 - cs4 * s_at) * 0.5
        l0, l1, l2 = lit[..., 0], lit[..., 1], lit[..., 2]
        Ec = (l0 * l1 * l2).sum(dim=2)
        mu3 = mu.view(ni, 1, 1)
        dl0 = (-cs4[..., 0] * 0.5) * l1 * l2
        dl1 = l0 * (-cs4[..., 1] * 0.5) * l2
        dl2 = l0 * l1 * (-cs4[..., 2] * 0.5)
        g = torch.zeros(ni, np_, n, device=device)
        for pos, dl in enumerate([dl0, dl1, dl2]):
            g.scatter_add_(2, cv4[..., pos], dl)
        g += mu3 * (-4.0 * s * (1.0 - s ** 2))
        return Ec + (mu3 * (1.0 - s**2)**2).sum(dim=2), Ec, g

    @torch.no_grad()
    def gradient_flow(self, s, cv, cs, mu, steps, dt=0.05, beta=0.9,
                      sched_t=None):
        ni, np_, n = self.ni, s.shape[1], self.n
        v = torch.zeros_like(s)
        plat = torch.zeros(ni, np_, device=device)
        bE = torch.full((ni, np_), float('inf'), device=device)
        decay_arr = 1.0 / (1.0 + 0.002 * torch.arange(steps, device=device, dtype=torch.float32))
        zeros_p = torch.zeros_like(plat)
        two_t = torch.tensor(2.0, device=device)
        one_t = torch.tensor(1.0, device=device)
        use_amp = device.type == 'cuda'

        for step in range(steps):
            mu_eff = mu * sched_t[step].item() if sched_t is not None else mu
            if use_amp:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    _, Ec, g = self.energy_and_grad(s, cv, cs, mu_eff)
                Ec, g = Ec.float(), g.float()
            else:
                _, Ec, g = self.energy_and_grad(s, cv, cs, mu_eff)

            imp = Ec < bE; bE = torch.where(imp, Ec, bE)
            plat = torch.where(imp, zeros_p, plat + 1)
            pm = plat >= 50; dec = decay_arr[step]
            gnorm = g.norm(dim=2, keepdim=True).clamp_(min=1e-10)
            dte = (dt * dec) / (1.0 + 0.05 * gnorm)
            pm3 = pm.unsqueeze(2)
            dte = dte * torch.where(pm3, two_t, one_t)
            gam = (Ec.clamp_(min=0) / (Ec + 1.0)).unsqueeze(2)
            v = beta * v - dte * (1.0 + gam) * g
            ns_val = 0.03 * dec
            ns = torch.where(pm3, ns_val * 4.0, ns_val)
            s = (s + v + torch.randn_like(s) * ns).clamp_(-1.0, 1.0)
        return s, bE

    @torch.no_grad()
    def solve_rate(self, s, cv, cs, mu):
        sr = torch.sign(s + 1e-10)
        _, E, _ = self.energy_and_grad(sr, cv, cs, mu)
        best = E.min(dim=1).values
        rate = (best < 0.5).float().mean().item()
        return rate, np.sqrt(rate * (1 - rate) / self.ni), best

    @torch.no_grad()
    def count_violations(self, s_best, cv, cs):
        """Returns per-instance violation count for rounded assignments."""
        ni, m, n = self.ni, self.m, self.n
        cv4 = cv.unsqueeze(1).expand(ni, 1, m, 3)
        s_ex = s_best.unsqueeze(1).unsqueeze(2).expand(ni, 1, m, n)
        s_at = torch.gather(s_ex, 3, cv4)
        cs4 = cs.unsqueeze(1).expand(ni, 1, m, 3)
        lit = (1.0 - cs4 * s_at) * 0.5
        Epc = (lit[..., 0] * lit[..., 1] * lit[..., 2]).squeeze(1)  # (ni, m)
        return (Epc > 0.1).sum(dim=1)  # (ni,) clause violations per instance


if HAS_COMPILE:
    try:
        BSDTSonarEngine.energy_and_grad = torch.compile(
            BSDTSonarEngine.energy_and_grad, mode='reduce-overhead')
        print('torch.compile ✓')
    except Exception as e:
        print(f'torch.compile skipped: {e}')


# ── Numba WalkSAT with flip counter ──
@njit(cache=True)
def walksat_numba(cv_np, cs_np, init, n, max_flips=100000, p_walk=0.57):
    m = cv_np.shape[0]; s = init.copy()
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for p in range(3): var_count[cv_np[c, p]] += 1
    max_occ = var_count.max() + 1
    var_clauses = np.full((n, max_occ), -1, dtype=np.int32)
    var_pos = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for p in range(3):
            v = cv_np[c, p]
            var_clauses[v, var_pos[v]] = c
            var_pos[v] += 1
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        cnt = 0
        for p in range(3):
            if cs_np[c, p] * s[cv_np[c, p]] > 0: cnt += 1
        clause_sat[c] = cnt
    unsat = np.empty(m, dtype=np.int32); n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0: unsat[n_unsat] = c; n_unsat += 1
    for flip in range(max_flips):
        if n_unsat == 0: return s, True, flip
        c = unsat[np.random.randint(n_unsat)]
        vs = np.array([cv_np[c,0], cv_np[c,1], cv_np[c,2]], dtype=np.int64)
        if np.random.random() < p_walk:
            chosen = vs[np.random.randint(3)]
        else:
            best_break = m + 1; chosen = vs[0]
            for vi in range(3):
                v = vs[vi]; bc = 0
                for ci in range(var_pos[v]):
                    cc = var_clauses[v, ci]
                    if clause_sat[cc] == 1:
                        for pp in range(3):
                            if cv_np[cc, pp] == v:
                                if cs_np[cc, pp] * s[v] > 0: bc += 1
                                break
                if bc < best_break: best_break = bc; chosen = v
        s[chosen] = -s[chosen]
        for ci in range(var_pos[chosen]):
            cc = var_clauses[chosen, ci]; old = clause_sat[cc]
            nc = 0
            for pp in range(3):
                if cs_np[cc, pp] * s[cv_np[cc, pp]] > 0: nc += 1
            clause_sat[cc] = nc
            if old > 0 and nc == 0: unsat[n_unsat] = cc; n_unsat += 1
            elif old == 0 and nc > 0:
                for ui in range(n_unsat):
                    if unsat[ui] == cc: unsat[ui] = unsat[n_unsat-1]; n_unsat -= 1; break
    return s, False, max_flips


def _ws_worker(args):
    inst, cv, cs, init, n = args
    res, ok, flips = walksat_numba(cv, cs, init, n)
    return inst, res, ok, flips


def walksat_parallel(s, cv, cs, mu, eng, max_workers=8):
    """Returns (s_out, n_recovered, flip_stats_per_instance)."""
    sr = torch.sign(s + 1e-10)
    _, E, _ = eng.energy_and_grad(sr, cv, cs, mu)
    bE = E.min(dim=1).values; fail = bE >= 0.5; bidx = E.argmin(dim=1)
    n_fail = fail.sum().item()
    if n_fail == 0: return s, 0, []

    cv_np = cv.cpu().numpy().astype(np.int64)
    cs_np = cs.cpu().numpy(); s_np = sr.cpu().numpy()
    s_out = s.clone(); n_ = eng.n
    work = [(i, cv_np[i], cs_np[i], s_np[i, bidx[i].item()], n_)
            for i in range(eng.ni) if fail[i]]

    rec = 0; flip_stats = []
    with ThreadPoolExecutor(max_workers=min(max_workers, len(work))) as pool:
        futs = {pool.submit(_ws_worker, w): w[0] for w in work}
        for f in as_completed(futs):
            inst, res, ok, flips = f.result()
            flip_stats.append({'inst': inst, 'ok': ok, 'flips': flips})
            if ok:
                rec += 1
                s_out[inst, 0] = torch.tensor(res, dtype=torch.float32, device=device)
    return s_out, rec, flip_stats


# Warm up Numba
print('JIT compiling...')
_ = walksat_numba(np.array([[0,1,2]], dtype=np.int64),
                  np.array([[1.,-1.,1.]]), np.array([1.,-1.,1.,-1.]), 4, 10)
print('Ready. Engine + delay70 + WalkSAT(Numba) loaded.')

In [ ]:
# ══════════════════════════════════════════════
# H9 SCALING TEST: n = 1500, 2000, 2500
# delay70 → WalkSAT, track violations + flips
# 50 instances × 1000 particles per size
# ══════════════════════════════════════════════

torch.manual_seed(42); np.random.seed(42)

NI, NP = 50, 1000
SIZES = [1500, 2000, 2500]

print('=' * 65)
print('H9 SCALING TEST — Does WalkSAT effort stay O(1)?')
print(f'Schedule: delay70 | {NI} instances × {NP} particles')
print('=' * 65)

# Include n=1000 baseline from H8a for comparison in final table
h9 = {1000: {'r1': 0.67, 'r_final': 1.00, 'viols_mean': 1.1,
             'viols_max': 3, 'flips_mean': 1.5, 'flips_max': 5,
             't_total': 379, 'ws_success': 33, 'ws_total': 33}}

for n in SIZES:
    steps = min(int(500 * np.sqrt(n)), 20000)
    eng = BSDTSonarEngine(n, ni=NI, np_=NP)
    cv, cs = eng.generate_instances()
    mu, _ = eng.compute_mu(cv)
    sched_t = precompute_schedule(sched_delay70, steps)

    print(f'\n{"═" * 65}')
    print(f'n={n}  m={eng.m}  steps={steps}')
    print(f'{"═" * 65}')

    # Stage 1: delay70 annealing
    if device.type == 'cuda': torch.cuda.empty_cache()
    s0 = torch.clamp(torch.randn(NI, NP, n, device=device) * 0.3, -0.9, 0.9)
    t0 = time.time()
    s1, _ = eng.gradient_flow(s0, cv, cs, mu, steps, sched_t=sched_t)
    t_anneal = time.time() - t0

    r1, se1, bE1 = eng.solve_rate(s1, cv, cs, mu)
    print(f'  Stage 1 (delay70):  {r1:6.1%} ± {se1:.1%}  ({t_anneal:.0f}s)')

    # Measure violations in failures
    sr = torch.sign(s1 + 1e-10)
    _, E, _ = eng.energy_and_grad(sr, cv, cs, mu)
    bidx = E.argmin(dim=1)
    s_best = sr[torch.arange(NI, device=device), bidx]
    viols = eng.count_violations(s_best, cv, cs)  # (NI,)
    fail_mask = bE1 >= 0.5
    n_fail = fail_mask.sum().item()

    if n_fail > 0:
        fail_viols = viols[fail_mask].cpu().numpy()
        v_mean = fail_viols.mean()
        v_max = int(fail_viols.max())
        v_median = float(np.median(fail_viols))
        print(f'  Failures: {n_fail}/{NI}')
        print(f'  Violations in failures: mean={v_mean:.1f}  '
              f'median={v_median:.0f}  max={v_max}')
    else:
        v_mean, v_max, v_median = 0, 0, 0
        print(f'  No failures! 100% at Stage 1.')

    # Stage 2: WalkSAT (skip nullspace — it wasn't helping)
    t0 = time.time()
    print(f'  WalkSAT on {n_fail} failures...')
    s2, rec, flip_stats = walksat_parallel(s1, cv, cs, mu, eng, max_workers=8)
    t_ws = time.time() - t0
    r2, se2, _ = eng.solve_rate(s2, cv, cs, mu)

    # Flip statistics
    if flip_stats:
        ok_flips = [fs['flips'] for fs in flip_stats if fs['ok']]
        fail_flips = [fs['flips'] for fs in flip_stats if not fs['ok']]
        f_mean = np.mean(ok_flips) if ok_flips else 0
        f_max = int(np.max(ok_flips)) if ok_flips else 0
        f_median = float(np.median(ok_flips)) if ok_flips else 0
        print(f'  WalkSAT: {rec}/{n_fail} recovered in {t_ws:.0f}s')
        print(f'  Flips (solved): mean={f_mean:.0f}  median={f_median:.0f}  max={f_max}')
        if fail_flips:
            print(f'  WalkSAT FAILED on {len(fail_flips)} instances (hit 100K flips)')
    else:
        f_mean, f_max = 0, 0

    t_total = t_anneal + t_ws
    print(f'  ─────────────────────────────────────')
    print(f'  FINAL n={n}: {r2:6.1%}   total={t_total:.0f}s')

    h9[n] = {
        'r1': r1, 'r_final': r2,
        'viols_mean': float(v_mean), 'viols_max': v_max,
        'flips_mean': float(f_mean), 'flips_max': f_max,
        't_total': t_total,
        'ws_success': rec, 'ws_total': n_fail,
        't_anneal': t_anneal, 't_ws': t_ws
    }

In [ ]:
# ══════════════════════════════════════════════
# RESULTS TABLE + SCALING CHARTS
# ══════════════════════════════════════════════

ALL_N = [1000, 1500, 2000, 2500]

print('=' * 80)
print('H9 SCALING RESULTS')
print('=' * 80)
print(f'  {"n":>5} | {"S1 Rate":>8} | {"Final":>6} | {"Viols":>6} | '
      f'{"Flips":>8} | {"WS ok":>7} | {"Time":>6}')
print('  ' + '-' * 65)
for n in ALL_N:
    r = h9[n]
    ws_str = f"{r.get('ws_success',0)}/{r.get('ws_total',0)}"
    print(f'  {n:>5} | {r["r1"]:>7.1%} | {r["r_final"]:>5.1%} | '
          f'{r["viols_mean"]:>5.1f} | {r["flips_mean"]:>7.0f} | '
          f'{ws_str:>7} | {r["t_total"]:>5.0f}s')

# Scaling verdict
print(f'\n  SCALING ANALYSIS:')
test_ns = [n for n in ALL_N if n >= 1500]
if all(h9[n]['r_final'] >= 0.99 for n in test_ns):
    print('  ★ 100% solve rate maintained at all tested sizes!')
else:
    for n in test_ns:
        if h9[n]['r_final'] < 0.99:
            print(f'  ✗ n={n}: solve rate dropped to {h9[n]["r_final"]:.1%}')

# Check if violations grow
v_list = [(n, h9[n]['viols_mean']) for n in ALL_N if h9[n]['viols_mean'] > 0]
if len(v_list) >= 2:
    ns_v, vs = zip(*v_list)
    if vs[-1] < vs[0] * 3:
        print(f'  ★ Violations stable: {vs[0]:.1f} → {vs[-1]:.1f} (not growing with n)')
    else:
        print(f'  ⚠ Violations growing: {vs[0]:.1f} → {vs[-1]:.1f}')

# Check if flips grow
f_list = [(n, h9[n]['flips_mean']) for n in ALL_N if h9[n]['flips_mean'] > 0]
if len(f_list) >= 2:
    ns_f, fs = zip(*f_list)
    if fs[-1] < 1000:
        print(f'  ★ WalkSAT flips O(1): {fs[0]:.0f} → {fs[-1]:.0f}')
    elif fs[-1] < fs[0] * (ns_f[-1]/ns_f[0])**2:
        print(f'  ◆ WalkSAT flips polynomial: {fs[0]:.0f} → {fs[-1]:.0f}')
    else:
        print(f'  ✗ WalkSAT flips may be exponential: {fs[0]:.0f} → {fs[-1]:.0f}')


# ── Charts ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('H9 Scaling Test — Odeyemi Olusegun Israel',
             fontsize=14, fontweight='bold')

# 1. Solve rates
ax = axes[0, 0]
r1s = [h9[n]['r1'] * 100 for n in ALL_N]
rfs = [h9[n]['r_final'] * 100 for n in ALL_N]
ax.plot(ALL_N, r1s, 'g--^', lw=2, ms=8, label='Stage 1 (delay70)')
ax.plot(ALL_N, rfs, 'r-o', lw=2.5, ms=9, label='Final (+WalkSAT)')
ax.set_xlabel('n'); ax.set_ylabel('Solve Rate (%)')
ax.set_title('Solve Rate vs n'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 108)

# 2. Violations per failure
ax = axes[0, 1]
vms = [h9[n]['viols_mean'] for n in ALL_N]
vmx = [h9[n]['viols_max'] for n in ALL_N]
ax.plot(ALL_N, vms, 'b-o', lw=2, ms=8, label='Mean violations')
ax.plot(ALL_N, vmx, 'b--s', lw=1.5, ms=6, alpha=0.6, label='Max violations')
ax.set_xlabel('n'); ax.set_ylabel('Clause violations')
ax.set_title('Violations in Failures vs n')
ax.legend(); ax.grid(True, alpha=0.3)

# 3. WalkSAT flips
ax = axes[0, 2]
fms = [h9[n]['flips_mean'] for n in ALL_N]
fmx = [h9[n]['flips_max'] for n in ALL_N]
ax.plot(ALL_N, fms, 'm-o', lw=2, ms=8, label='Mean flips')
ax.plot(ALL_N, fmx, 'm--s', lw=1.5, ms=6, alpha=0.6, label='Max flips')
ax.set_xlabel('n'); ax.set_ylabel('WalkSAT flips')
ax.set_title('WalkSAT Effort vs n')
ax.legend(); ax.grid(True, alpha=0.3)

# 4. Stage 1 rate decay
ax = axes[1, 0]
ax.plot(ALL_N, r1s, 'g-o', lw=2, ms=8)
# Fit exponential decay
from numpy.polynomial import polynomial as P
ln_r1 = [np.log(max(r/100, 0.01)) for r in r1s]
coeffs = np.polyfit(ALL_N, ln_r1, 1)
fit_n = np.linspace(1000, 2500, 100)
fit_r = np.exp(np.polyval(coeffs, fit_n)) * 100
ax.plot(fit_n, fit_r, 'k--', alpha=0.5,
        label=f'Fit: exp({coeffs[0]:.5f}·n)')
half_life = -np.log(2) / coeffs[0] if coeffs[0] < 0 else float('inf')
ax.set_xlabel('n'); ax.set_ylabel('Stage 1 Rate (%)')
ax.set_title(f'Stage 1 Decay (half-life={half_life:.0f})')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 108)

# 5. Runtime scaling
ax = axes[1, 1]
ts = [h9[n]['t_total'] for n in ALL_N]
ax.plot(ALL_N, ts, 'k-o', lw=2, ms=8, label='Total time')
# Reference O(n^2.5) line
ref = [ts[0] * (n / ALL_N[0]) ** 2.5 for n in ALL_N]
ax.plot(ALL_N, ref, 'k--', alpha=0.4, label='O(n^2.5) reference')
ax.set_xlabel('n'); ax.set_ylabel('Time (s)')
ax.set_title('Runtime Scaling')
ax.legend(); ax.grid(True, alpha=0.3)

# 6. Summary bar at largest n
ax = axes[1, 2]
n_max = ALL_N[-1]
if n_max in h9:
    r = h9[n_max]
    stgs = ['Stage 1', 'Final']
    vals = [r['r1'] * 100, r['r_final'] * 100]
    cols = ['green', 'red']
    bars = ax.bar(stgs, vals, color=cols, alpha=0.85, width=0.4)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 1,
                f'{v:.0f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_title(f'n={n_max}: Final Result')
ax.set_ylabel('Solve Rate (%)'); ax.set_ylim(0, 115)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('h9_scaling_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h9_scaling_results.png')